# 02 — Text Cleaning
Clean and normalise Arabic complaint text.

> **Run `00_setup.ipynb` first** so the repo and packages are available.

In [12]:
# Core
import os
import re
import pandas as pd

## 1) Paths & Loading

We read input CSVs from the cloned repo inside Colab runtime.
We save outputs to Google Drive so results persist after the session ends.

In [27]:
# --- Repo paths (read-only source inside Colab runtime) ---
REPO_DIR = "/content/NLP-complaints-system"
DATA_DIR = os.path.join(REPO_DIR, "data", "text")

labeled_path   = os.path.join(DATA_DIR, "complaints_labeled.csv")
unlabeled_path = os.path.join(DATA_DIR, "complaints_unlabeled.csv")

# --- Output on Drive (persistent) ---
# If you are using MyDrive:
DRIVE_OUT_DIR = "/content/drive/MyDrive/NLP-Complaints-Team/data/processed"

# If you are using a Shared Drive, it is usually:
# DRIVE_OUT_DIR = "/content/drive/Shareddrives/<SharedDriveName>/NLP-Complaints-Team/data/processed"

os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

# --- Load data ---
df_labeled = pd.read_csv(labeled_path)
df_unlabeled = pd.read_csv(unlabeled_path)

print("Loaded labeled:", df_labeled.shape, "| unlabeled:", df_unlabeled.shape)
print("Output dir:", DRIVE_OUT_DIR)

Loaded labeled: (651, 2) | unlabeled: (6146, 1)
Output dir: /content/drive/MyDrive/NLP-Complaints-Team/data/processed


## 2) Quick Inspection

We verify column names and basic null ratios before cleaning.

In [28]:
print("Labeled columns:", df_labeled.columns.tolist())
print("Unlabeled columns:", df_unlabeled.columns.tolist())

TEXT_COL = "text"  # expected column name in both files

print("Nulls (labeled text):", df_labeled[TEXT_COL].isna().mean())
print("Nulls (unlabeled text):", df_unlabeled[TEXT_COL].isna().mean())

df_labeled.head()

Labeled columns: ['source_label', 'text']
Unlabeled columns: ['text']
Nulls (labeled text): 0.0
Nulls (unlabeled text): 0.0


,source_label,text
0,خدمة_عملاء,تواصلت مع خدمة العملاء عدة مرات ولم يحل أحد مش...
1,خدمة_عملاء,موظف خدمة العملاء كان غير محترم في تعامله معي
2,خدمة_عملاء,انتظرت على الخط الساخن أكثر من ساعة ولم يرد أحد
3,خدمة_عملاء,الردود التي تلقيتها من فريق الدعم كانت غير مفي...
4,خدمة_عملاء,لم أتلق أي رد على رسالتي الإلكترونية منذ أسبوع...


## 3) Arabic Cleaning Utilities

We define:
- diacritics (tashkeel) removal
- tatweel removal
- URL/email/mentions/hashtags removal
- normalization of common Arabic letter variants
- removing non-Arabic characters (digits, latin, punctuation, emojis)
- reducing repeated characters
- whitespace normalization

In [29]:
# ==========================
# Arabic text cleaning utils
# ==========================

# Arabic diacritics + Qur'anic marks (tashkeel)
AR_DIACRITICS_RE = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670\u06D6-\u06ED]")

# Tatweel
TATWEEL = "\u0640"

# Social noise
URL_RE = re.compile(r"(https?://\S+|www\.\S+)")
EMAIL_RE = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#\w+")

# Keep Arabic letters + whitespace only (drops digits/latin/punct/emojis)
NON_ARABIC_RE = re.compile(r"[^\u0600-\u06FF\s]")

# Reduce long repeats: جمييييل -> جمييل (keep 2)
REPEATS_RE = re.compile(r"(.)\1{2,}")

# Normalize spaces
MULTISPACE_RE = re.compile(r"\s+")


def normalize_arabic(text: str) -> str:
    """Normalize Arabic letters and remove diacritics."""
    # Remove diacritics
    text = AR_DIACRITICS_RE.sub("", text)

    # Remove tatweel
    text = text.replace(TATWEEL, "")

    # Normalize Alef variants -> ا
    text = re.sub(r"[إأآا]", "ا", text)

    # Normalize Alif maqsurah -> ي
    text = text.replace("ى", "ي")

    # Normalize taa marbuta -> ه (common choice; keep consistent)
    text = text.replace("ة", "ه")

    return text


def clean_arabic(text) -> str:
    """Full cleaning pipeline for Arabic complaint text."""
    if not isinstance(text, str):
        return ""

    text = text.strip()

    # Remove social noise
    text = URL_RE.sub(" ", text)
    text = EMAIL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = HASHTAG_RE.sub(" ", text)

    # Normalize Arabic
    text = normalize_arabic(text)

    # Remove non-Arabic chars (digits, punctuation, latin, emojis, etc.)
    text = NON_ARABIC_RE.sub(" ", text)

    # Reduce repetitions
    text = REPEATS_RE.sub(r"\1\1", text)

    # Normalize whitespace
    text = MULTISPACE_RE.sub(" ", text).strip()

    return text

## 4) Apply Cleaning

We apply the cleaning function to the `text` column in both datasets.
We keep copies so we do not mutate the original dataframes.

In [30]:
df_labeled_clean = df_labeled.copy()
df_unlabeled_clean = df_unlabeled.copy()

df_labeled_clean[TEXT_COL] = df_labeled_clean[TEXT_COL].map(clean_arabic)
df_unlabeled_clean[TEXT_COL] = df_unlabeled_clean[TEXT_COL].map(clean_arabic)

print("Done cleaning.")

Done cleaning.


## 5) Drop Empty Texts (Post-clean)

Sometimes a row becomes empty after cleaning (e.g., it was only symbols/links).
We drop empty strings to avoid hurting modeling quality.

In [31]:
def drop_empty_texts(df: pd.DataFrame, text_col: str) -> pd.DataFrame:
    """Drop rows where the cleaned text is empty/whitespace."""
    df = df.copy()
    df[text_col] = df[text_col].fillna("").astype(str)
    before = len(df)
    df = df[df[text_col].str.strip().ne("")].copy()
    after = len(df)
    print(f"Kept {after}/{before} rows (dropped {before - after} empty texts).")
    return df


print("Labeled:")
df_labeled_clean = drop_empty_texts(df_labeled_clean, TEXT_COL)

print("Unlabeled:")
df_unlabeled_clean = drop_empty_texts(df_unlabeled_clean, TEXT_COL)

Labeled:
Kept 651/651 rows (dropped 0 empty texts).
Unlabeled:
Kept 6106/6146 rows (dropped 40 empty texts).


## 6) Save Cleaned CSVs (Drive)

We save cleaned versions to Drive under `data/processed/`.
Do NOT commit these CSV files to GitHub.

In [32]:
out_labeled = os.path.join(DRIVE_OUT_DIR, "complaints_labeled_clean.csv")
out_unlabeled = os.path.join(DRIVE_OUT_DIR, "complaints_unlabeled_clean.csv")

df_labeled_clean.to_csv(out_labeled, index=False)
df_unlabeled_clean.to_csv(out_unlabeled, index=False)

print("Saved:", out_labeled, "exists?", os.path.exists(out_labeled))
print("Saved:", out_unlabeled, "exists?", os.path.exists(out_unlabeled))

Saved: /content/drive/MyDrive/NLP-Complaints-Team/data/processed/complaints_labeled_clean.csv exists? True
Saved: /content/drive/MyDrive/NLP-Complaints-Team/data/processed/complaints_unlabeled_clean.csv exists? True


## 7) Sanity Checks (Before/After)

We preview a few samples and confirm the output files exist.

In [33]:
# Show before/after examples (first 5)
sample_idx = [0, 1, 2, 3, 4]
preview = pd.DataFrame({
    "before": df_labeled.loc[sample_idx, TEXT_COL].astype(str).tolist(),
    "after": df_labeled_clean.loc[sample_idx, TEXT_COL].astype(str).tolist(),
})
preview

,before,after
0,تواصلت مع خدمة العملاء عدة مرات ولم يحل أحد مش...,تواصلت مع خدمه العملاء عده مرات ولم يحل احد مش...
1,موظف خدمة العملاء كان غير محترم في تعامله معي,موظف خدمه العملاء كان غير محترم في تعامله معي
2,انتظرت على الخط الساخن أكثر من ساعة ولم يرد أحد,انتظرت علي الخط الساخن اكثر من ساعه ولم يرد احد
3,الردود التي تلقيتها من فريق الدعم كانت غير مفي...,الردود التي تلقيتها من فريق الدعم كانت غير مفي...
4,لم أتلق أي رد على رسالتي الإلكترونية منذ أسبوع...,لم اتلق اي رد علي رسالتي الالكترونيه منذ اسبوع...
